# Notebook 03: LayoutLM Multimodal Fine-Tuning (Text + 2D Layout)

**Objective:** Fine-tune `microsoft/layoutlm-base-uncased` with WordPiece tokens and normalized 2D spatial bounding boxes $[x_0, y_0, x_1, y_1]$ on SROIE receipts.

**Research Question:**
$$\Delta F1 = F1_{\text{LayoutLM}} - F1_{\text{BERT}}$$

*(Can be executed locally or on Google Colab / Kaggle with free T4 GPU)*

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.train_layoutlm import train_layoutlm

# Train and evaluate LayoutLM-base with 2D coordinates
layoutlm_results = train_layoutlm(
    train_file=str(project_root / "data" / "processed" / "train.json"),
    test_file=str(project_root / "data" / "processed" / "test.json"),
    output_path=str(project_root / "results" / "layoutlm_results.json"),
    bert_results_path=str(project_root / "results" / "bert_results.json"),
    epochs=5,
    batch_size=8,
    lr=5e-5,
)

## Generate Visual Comparison & Benchmark Plot

In [ ]:
from src.visualize import generate_svg_chart, load_json_metrics

regex_m = load_json_metrics(str(project_root / "results" / "regex_baseline.json"))
bert_m = load_json_metrics(str(project_root / "results" / "bert_results.json"))
layout_m = load_json_metrics(str(project_root / "results" / "layoutlm_results.json"))

benchmark_data = {}
if regex_m:
    benchmark_data["OCR + Regex"] = {k: v["f1"] for k, v in regex_m["metrics"].items() if "f1" in v}
if bert_m:
    benchmark_data["BERT (Text)"] = {k: v["f1"] for k, v in bert_m["metrics"].items() if "f1" in v}
if layout_m:
    benchmark_data["LayoutLM"] = {k: v["f1"] for k, v in layout_m["metrics"].items() if "f1" in v}

generate_svg_chart(benchmark_data, str(project_root / "results" / "f1_comparison.svg"))
print("Benchmark chart updated!")